In [0]:
%run ../config/config

In [0]:
spark.conf.set("spark.databricks.execution.timeout", "18000")

In [0]:
STAGING = f"{CATALOG}.{STAGING_SCHEMA}.{ZEROBUS_STAGING}"
BRONZE = f"{CATALOG}.{BRONZE_SCHEMA}.{ZEROBUS_BRONZE}"

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {BRONZE}
    (
        event_id STRING,
        source_updated BIGINT,
        ingested_at BIGINT,
        event_json STRING
    )
    USING DELTA
""")

In [0]:
spark.sql(f"""
MERGE INTO {BRONZE} AS target
USING (
    SELECT
        event_id,
        source_updated,
        ingested_at,
        event_json
    FROM (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY event_id
                ORDER BY source_updated DESC, ingested_at DESC
            ) AS rn
        FROM {STAGING}
    )
    WHERE rn = 1
) AS source

ON target.event_id = source.event_id

WHEN MATCHED AND source.source_updated > target.source_updated
THEN UPDATE SET
    target.source_updated = source.source_updated,
    target.ingested_at = source.ingested_at,
    target.event_json = source.event_json

WHEN NOT MATCHED THEN
    INSERT (
        event_id,
        source_updated,
        ingested_at,
        event_json
    )
    VALUES (
        source.event_id,
        source.source_updated,
        source.ingested_at,
        source.event_json
    );
""")